# 🧬 Automation Pipeline: AMR Phenotype Prediction

Welcome to the automated pipeline interface. This notebook directly mirrors the application background architecture, allowing you to generate predictions, process cleanings, and render diagnostic reports quickly.

---
### ⚠️ Instructions:
1. **Just edit the `input_csv_path` variable in the 3rd cell down below** to point to your dataset.
2. Run all cells (`Run All`). The pipeline will automatically activate, executing all data preprocessing and cleaning algorithms.
3. The final XGBoost predictions will be saved directly into a CSV output file alongside a performance evaluation!

> **Note:** Similar to the Streamlit app, if your input data provides the target variable as `null` for some instances (or lacks it entirely), this pipeline will still predict phenotypes for **all** rows. However, for the final Classification Report, it will filter and consider **only** those values which safely possess a valid ground-truth target variable.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, 
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

# Force working directory to project root so Pickled models load securely
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Import our deployed preprocessing pipeline logic
sys.path.append('app')
import app

In [ ]:
# ==========================================================================
# 💾 EDIT THIS PATH TO YOUR INPUT DATASET 
# ==========================================================================
input_csv_path = "BVBRC_genome_amr.csv"

output_csv_path = "amr_pipeline_predictions.csv"

## 1. Data Preprocessing & Cleaning

In [ ]:
print(f"Loading data from {input_csv_path}...")
df = pd.read_csv(input_csv_path)

# Identify whether the inference set has the target variable present at all
has_target = False
if "Resistant Phenotype" in df.columns:
    has_target = True
    print("Target variable 'Resistant Phenotype' found. Classification metrics will be generated on valid rows.")
else:
    print("No 'Resistant Phenotype' column found. The pipeline will operate in pure-prediction mode.")

# Clean intermediate variables immediately if target is present
if has_target:
    df = df[df['Resistant Phenotype'] != 'Intermediate'].copy()
    
df = df.drop_duplicates()

# Send inputs to deployed processing infrastructure
X, y_true, genome_ids = app.preprocess(df, has_target=has_target)
print("Data safely preprocessed into Target Encoders, One-Hot Encoded variables, and Ordinal mappings.")

## 2. Ingest Predictions & Save

In [ ]:
print("Evaluating via XGBoost Engine...")
y_labels, y_probs = app.predict(X)

# Construct outputs
out_df = pd.DataFrame({
    "Genome ID": genome_ids,
    "Antibiotic": df["Antibiotic"].values,
    "Predicted Phenotype": y_labels,
    "Resistance Probability": np.round(y_probs, 4),
})

if has_target:
    out_df.insert(2, "Actual Phenotype", df["Resistant Phenotype"].values)

out_df.to_csv(output_csv_path, index=False)
print(f"\u2705 Final predictions and probability arrays successfully saved to `{output_csv_path}`!")
display(out_df.head(10))

## 3. Model Classification Report
If applicable, we compare model performance on exclusively those rows possessing empirical outcome limits.

In [ ]:
if has_target:
    valid_mask = y_true.notna()
    y_eval_true = y_true[valid_mask]
    y_eval_prob = y_probs[valid_mask]
    # Binarize output probability against optimum threshold
    y_eval_pred = (y_eval_prob >= app.CUSTOM_THRESHOLD).astype(int)
    
    if len(y_eval_true) > 0:
        acc = accuracy_score(y_eval_true, y_eval_pred)
        f1_w = f1_score(y_eval_true, y_eval_pred, average='weighted')
        auc = roc_auc_score(y_eval_true, y_eval_prob)
        
        print("=" * 45)
        print(f"Accuracy:           {acc:.4f}")
        print(f"Weighted F1-Score:  {f1_w:.4f}")
        print(f"AUC-ROC:            {auc:.4f}")
        print(f"Active Threshold:   {app.CUSTOM_THRESHOLD}")
        print("=" * 45)
        
        print("\n--- Classification Report ---")
        print(classification_report(y_eval_true, y_eval_pred, target_names=["Susceptible (0)", "Resistant (1)"]))
        
        # Visualise Matrix natively
        cm = confusion_matrix(y_eval_true, y_eval_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Susceptible", "Resistant"])
        disp.plot(ax=ax, colorbar=False, cmap="Blues")
        plt.grid(False)
        plt.title("Confusion Matrix")
        plt.show()
    else:
        print("\u26a0\ufe0f Target variables found, but all instances were `null` or ambiguous. Cannot formulate final testing metrics.")
else:
    print("\u26a0\ufe0f Target column globally absent. Metric evaluations successfully skipped.")